# AI Precision Oncology

# Notebook 03

## Clinical Feature Engineering & Data Preprocessing

---

### Research Question

How can heterogeneous clinical data be transformed into machine-learning-ready features while preserving clinical meaning and ensuring reproducibility?

---

### Objectives

- Assess feature quality.
- Design a missing-data strategy.
- Engineer clinically meaningful variables.
- Build reusable preprocessing pipelines.
- Export preprocessing objects for downstream AI models.

In [1]:
# ============================================================
# Imports
# ============================================================

from pathlib import Path

import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import (
    SimpleImputer,
    KNNImputer
)

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    OrdinalEncoder
)

from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# Load Dataset
# ============================================================

PROJECT_ROOT = Path.cwd().parent

DATASET = (
    PROJECT_ROOT
    / "datasets"
    / "Breast Cancer METABRIC.csv"
)

df = pd.read_csv(DATASET)

print("=" * 60)
print("Dataset Loaded")
print("=" * 60)

print(df.shape)

Dataset Loaded
(2509, 34)


In [3]:
# ============================================================
# Data Audit
# ============================================================

audit = pd.DataFrame()

audit["Data Type"] = df.dtypes

audit["Missing"] = df.isnull().sum()

audit["Missing %"] = (
    df.isnull().mean() * 100
).round(2)

audit["Unique"] = df.nunique()

audit["Example"] = df.iloc[0]

display(audit)

,Data Type,Missing,Missing %,Unique,Example
Patient ID,object,0,0.00,2509,MB-0000
Age at Diagnosis,float64,11,0.44,1843,75.65
Type of Breast Surgery,object,554,22.08,2,Mastectomy
Cancer Type,object,0,0.00,2,Breast Cancer
Cancer Type Detailed,object,0,0.00,8,Breast Invasive Ductal Carcinoma
Cellularity,object,592,23.60,3,NaN
Chemotherapy,object,529,21.08,2,No
Pam50 + Claudin-low subtype,object,529,21.08,7,claudin-low
Cohort,float64,11,0.44,9,1.0
ER status measured by IHC,object,83,3.31,2,Positve


In [4]:
# ============================================================
# Feature Selection
# ============================================================

# -------------------------
# Target Variables
# -------------------------

TARGETS = [
    "Overall Survival Status",
    "Overall Survival (Months)"
]

# -------------------------
# Remove (Data Leakage / IDs)
# -------------------------

DROP_COLUMNS = [
    "Patient ID",
    "Patient's Vital Status",
    "Relapse Free Status",
    "Relapse Free Status (Months)"
]

# -------------------------
# Numerical Features
# -------------------------

NUMERICAL_FEATURES = [

    "Age at Diagnosis",

    "Tumor Size",

    "Tumor Stage",

    "Neoplasm Histologic Grade",

    "Lymph nodes examined positive",

    "Mutation Count",

    "Nottingham prognostic index"
]

# -------------------------
# Categorical Features
# -------------------------

CATEGORICAL_FEATURES = [

    "Type of Breast Surgery",

    "Cancer Type",

    "Cancer Type Detailed",

    "Cellularity",

    "Chemotherapy",

    "ER Status",

    "HER2 Status",

    "Hormone Therapy",

    "PR Status",

    "Radio Therapy",

    "Tumor Other Histologic Subtype",

    "Primary Tumor Laterality",

    "Inferred Menopausal State"
]

print("=" * 60)
print("Feature Categories")
print("=" * 60)

print(f"Target Variables      : {len(TARGETS)}")
print(f"Numerical Features    : {len(NUMERICAL_FEATURES)}")
print(f"Categorical Features  : {len(CATEGORICAL_FEATURES)}")
print(f"Dropped Columns       : {len(DROP_COLUMNS)}")

Feature Categories
Target Variables      : 2
Numerical Features    : 7
Categorical Features  : 13
Dropped Columns       : 4


In [5]:
# ============================================================
# Project Configuration
# ============================================================

FEATURE_CONFIG = {

    "target": TARGETS,

    "drop": DROP_COLUMNS,

    "numerical": NUMERICAL_FEATURES,

    "categorical": CATEGORICAL_FEATURES

}

print("=" * 60)
print("Project Configuration")
print("=" * 60)

for key, value in FEATURE_CONFIG.items():

    print(f"{key.upper():15}: {len(value)} variables")

Project Configuration
TARGET         : 2 variables
DROP           : 4 variables
NUMERICAL      : 7 variables
CATEGORICAL    : 13 variables


In [6]:
# ============================================================
# Missing Data Strategy
# ============================================================

NUMERICAL_IMPUTER = SimpleImputer(
    strategy="median"
)

CATEGORICAL_IMPUTER = SimpleImputer(
    strategy="most_frequent"
)

print("=" * 60)
print("Imputation Strategy")
print("=" * 60)

print("Numerical Features  -> Median")
print("Categorical Features -> Most Frequent")

Imputation Strategy
Numerical Features  -> Median
Categorical Features -> Most Frequent


In [7]:
# ============================================================
# Numerical Pipeline
# ============================================================

numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            NUMERICAL_IMPUTER
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

print("Numerical preprocessing pipeline created.")

Numerical preprocessing pipeline created.


In [8]:
# ============================================================
# Categorical Pipeline
# ============================================================

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            CATEGORICAL_IMPUTER
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

print("Categorical preprocessing pipeline created.")

Categorical preprocessing pipeline created.


In [9]:
# ============================================================
# Complete Preprocessing Pipeline
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline,
            NUMERICAL_FEATURES
        ),
        (
            "cat",
            categorical_pipeline,
            CATEGORICAL_FEATURES
        )
    ]
)

print("=" * 60)
print("Complete Preprocessing Pipeline Created")
print("=" * 60)

Complete Preprocessing Pipeline Created


In [10]:
# ============================================================
# Prepare Dataset for Machine Learning
# ============================================================

ml_df = df.drop(columns=DROP_COLUMNS)

ml_df = ml_df.dropna(
    subset=["Overall Survival Status"]
)

X = ml_df.drop(columns=TARGETS)

y = (
    ml_df["Overall Survival Status"]
    .map({
        "Living": 0,
        "Deceased": 1
    })
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 60)
print("Train/Test Split")
print("=" * 60)

print(f"Training Samples : {len(X_train)}")
print(f"Testing Samples  : {len(X_test)}")

Train/Test Split
Training Samples : 1584
Testing Samples  : 397


In [11]:
# ============================================================
# Fit Preprocessing Pipeline
# ============================================================

print("=" * 70)
print("Fitting Preprocessing Pipeline")
print("=" * 70)

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Pipeline fitted successfully.")

print()

print(f"Training Matrix Shape : {X_train_processed.shape}")
print(f"Testing Matrix Shape  : {X_test_processed.shape}")

Fitting Preprocessing Pipeline
Pipeline fitted successfully.

Training Matrix Shape : (1584, 46)
Testing Matrix Shape  : (397, 46)


In [12]:
# ============================================================
# Feature Names After Transformation
# ============================================================

feature_names = preprocessor.get_feature_names_out()

print("=" * 70)
print("Generated Features")
print("=" * 70)

print(f"Total Features : {len(feature_names)}")

print()

for feature in feature_names:
    print(feature)

Generated Features
Total Features : 46

num__Age at Diagnosis
num__Tumor Size
num__Tumor Stage
num__Neoplasm Histologic Grade
num__Lymph nodes examined positive
num__Mutation Count
num__Nottingham prognostic index
cat__Type of Breast Surgery_Breast Conserving
cat__Type of Breast Surgery_Mastectomy
cat__Cancer Type_Breast Cancer
cat__Cancer Type_Breast Sarcoma
cat__Cancer Type Detailed_Breast
cat__Cancer Type Detailed_Breast Angiosarcoma
cat__Cancer Type Detailed_Breast Invasive Ductal Carcinoma
cat__Cancer Type Detailed_Breast Invasive Lobular Carcinoma
cat__Cancer Type Detailed_Breast Invasive Mixed Mucinous Carcinoma
cat__Cancer Type Detailed_Breast Mixed Ductal and Lobular Carcinoma
cat__Cancer Type Detailed_Invasive Breast Carcinoma
cat__Cancer Type Detailed_Metaplastic Breast Cancer
cat__Cellularity_High
cat__Cellularity_Low
cat__Cellularity_Moderate
cat__Chemotherapy_No
cat__Chemotherapy_Yes
cat__ER Status_Negative
cat__ER Status_Positive
cat__HER2 Status_Negative
cat__HER2 Statu

In [15]:
# ============================================================
# Preview Processed Features (Robust Version)
# ============================================================

if hasattr(X_train_processed, "toarray"):
    processed_array = X_train_processed.toarray()
else:
    processed_array = X_train_processed

processed_df = pd.DataFrame(
    processed_array,
    columns=feature_names
)

print("=" * 70)
print("Processed Dataset")
print("=" * 70)

display(processed_df.head())

Processed Dataset


,num__Age at Diagnosis,num__Tumor Size,num__Tumor Stage,num__Neoplasm Histologic Grade,num__Lymph nodes examined positive,num__Mutation Count,num__Nottingham prognostic index,cat__Type of Breast Surgery_Breast Conserving,cat__Type of Breast Surgery_Mastectomy,cat__Cancer Type_Breast Cancer,...,cat__Tumor Other Histologic Subtype_Medullary,cat__Tumor Other Histologic Subtype_Metaplastic,cat__Tumor Other Histologic Subtype_Mixed,cat__Tumor Other Histologic Subtype_Mucinous,cat__Tumor Other Histologic Subtype_Other,cat__Tumor Other Histologic Subtype_Tubular/ cribriform,cat__Primary Tumor Laterality_Left,cat__Primary Tumor Laterality_Right,cat__Inferred Menopausal State_Post,cat__Inferred Menopausal State_Pre
0,-0.828135,-0.025139,0.347230,-0.622500,-0.490648,0.584990,-0.818710,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,-1.671846,-0.657844,0.347230,0.955165,-0.229233,-0.907158,0.882571,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
2,-0.501187,-0.278221,-1.444342,-0.622500,-0.490648,0.584990,-0.825584,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,0.936763,0.227943,0.347230,-0.622500,-0.490648,1.082373,-2.530302,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4,-1.190722,-0.404762,-1.444342,-0.622500,-0.490648,3.071903,-0.829021,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
